# GPT (Generative Pretrained Transformer)
This notebook implements GPT with Causal Language Modeling (CLM) task for text generation.

## Imports and Dataset

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import (
    TextVectorization,
    Embedding,
    Dense,
    LayerNormalization,
    MultiHeadAttention
)
from tensorflow.keras import Model

data = [
    "the quick brown fox jumps over the lazy dog",
    "machine learning is a subset of artificial intelligence",
    "transformers have revolutionized natural language processing",
    "attention is all you need for deep learning",
    "gpt is a generative pretrained transformer model",
    "autoregressive language modeling predicts the next token",
    "gpt performs well on many downstream nlp tasks",
    "pretraining on large corpora improves model performance",
    "fine tuning gpt on specific tasks yields great results",
    "language generation with transformers is very powerful"
]

# Add start and end tokens
start_token = "start"
end_token = "end"
data = [start_token + " " + x + " " + end_token for x in data]

## Tokenization

In [ ]:
vocab_size = 500
sequence_length = 20

text_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length
)

text_vectorization.adapt(data)

tokens = text_vectorization(data)

# Create input-target pairs for next token prediction
input_sequences = tokens[:, :-1]
target_sequences = tokens[:, 1:]

## Positional Embedding Layer

In [ ]:
class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim):
        super().__init__()

        self.token_embedding = Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )

        self.position_embedding = Embedding(
            input_dim=sequence_length,
            output_dim=embed_dim
        )

    def call(self, inputs):
        length = tf.shape(inputs)[-1]

        positions = tf.range(
            start=0,
            limit=length,
            delta=1
        )

        token_embeddings = self.token_embedding(inputs)
        position_embeddings = self.position_embedding(positions)

        return token_embeddings + position_embeddings

## Transformer Decoder Layer (Causal)

In [ ]:
class TransformerDecoderLayer(tf.keras.layers.Layer):

    def __init__(self, embed_dim, dense_dim, num_heads):
        super().__init__()

        self.causal_attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            Dense(dense_dim, activation="relu"),
            Dense(embed_dim)
        ])

        self.layernorm1 = LayerNormalization()
        self.layernorm2 = LayerNormalization()

    def call(self, inputs):
        attention_output = self.causal_attention(
            query=inputs,
            value=inputs,
            key=inputs,
            use_causal_mask=True
        )

        out1 = self.layernorm1(
            inputs + attention_output
        )

        ffn_output = self.ffn(out1)

        return self.layernorm2(
            out1 + ffn_output
        )

## Build GPT Model

In [ ]:
embed_dim = 128
dense_dim = 256
num_heads = 4
num_layers = 2

decoder_input = tf.keras.Input(
    shape=(sequence_length - 1,),
    dtype="int64"
)

embeddings = PositionalEmbedding(
    sequence_length,
    vocab_size,
    embed_dim
)(decoder_input)

decoder_output = embeddings

for _ in range(num_layers):
    decoder_output = TransformerDecoderLayer(
        embed_dim,
        dense_dim,
        num_heads
    )(decoder_output)

lm_output = Dense(
    vocab_size,
    activation="softmax"
)(decoder_output)

gpt = Model(
    decoder_input,
    lm_output
)

## Compile Model

In [ ]:
gpt.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

gpt.summary()

## Train Model

In [ ]:
history = gpt.fit(
    input_sequences,
    target_sequences,
    epochs=30,
    batch_size=2,
    verbose=1
)

## Final Metrics

In [ ]:
final_acc = history.history['accuracy'][-1]
final_loss = history.history['loss'][-1]

print("\nFinal Accuracy:", round(final_acc * 100, 2), "%")
print("Final Loss:", round(final_loss, 4))

## Inference - Text Generation

In [ ]:
def generate_text(seed_text, num_tokens=5):
    """Generate text by predicting next tokens"""
    vocab = text_vectorization.get_vocabulary()
    
    generated = seed_text
    seed_tokens = text_vectorization([seed_text]).numpy()[0, :num_tokens]
    
    for _ in range(num_tokens):
        input_tokens = tf.constant([seed_tokens], dtype=tf.int64)
        prediction = gpt.predict(input_tokens, verbose=0)
        
        # Get last token prediction
        last_pred = prediction[0, -1, :]
        next_token_id = tf.argmax(last_pred).numpy()
        
        if next_token_id < len(vocab):
            next_word = vocab[next_token_id]
            generated += " " + next_word
        
        # Update seed_tokens for next iteration
        seed_tokens = tf.concat([seed_tokens[1:], [next_token_id]], axis=0).numpy()
    
    return generated

## Generate Text

In [ ]:
seed = "start the quick"
generated_text = generate_text(seed, num_tokens=5)

print("\nGenerated Text:")
print(generated_text)